# T11I Moisture Bin Interpretability

???bin??? attribution / occlusion / FSP?? / task???????????


In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess
import sys

GITHUB_REPO = '2Kentaro1/wood-moisture-2d-cnn'
PROJECT_DIR = Path('/content/wood-moisture-2d-cnn')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/wood-moisture-2d-cnn-outputs')
REPO_URL = f'https://github.com/{GITHUB_REPO}.git'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    os.environ['OUTPUT_DIR'] = str(DRIVE_OUTPUT_DIR)
except Exception as exc:
    print(f'Google Drive mount skipped: {exc}')
    os.environ.setdefault('OUTPUT_DIR', 'outputs')

token = os.environ.get('GITHUB_TOKEN', '').strip()
if not token:
    token = getpass('GitHub token (empty for public repo): ').strip()
clone_url = REPO_URL if not token else f'https://{token}@github.com/{GITHUB_REPO}.git'

if PROJECT_DIR.exists() and (PROJECT_DIR / '.git').exists():
    os.chdir(PROJECT_DIR)
    try:
        if token:
            subprocess.run(['git', 'pull', clone_url, 'main'], check=True)
            subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], check=False)
        else:
            subprocess.run(['git', 'pull'], check=True)
    except subprocess.CalledProcessError as exc:
        print(f'git pull skipped/failed: {exc}')
else:
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    if PROJECT_DIR.exists() and not (PROJECT_DIR / '.git').exists():
        raise RuntimeError(f'{PROJECT_DIR} exists but is not a git repository. Rename/remove it or set PROJECT_DIR.')
    subprocess.run(['git', 'clone', clone_url, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], check=False)

os.environ['PROJECT_ROOT'] = str(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
Path(os.environ['OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT={os.environ["PROJECT_ROOT"]}')
print(f'OUTPUT_DIR={os.environ["OUTPUT_DIR"]}')
print('src exists:', (PROJECT_DIR / 'src').exists())
print('train exists:', (PROJECT_DIR / 'data' / 'train.csv').exists())
print('test exists:', (PROJECT_DIR / 'data' / 'test.csv').exists())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
import os
from pathlib import Path
MOISTURE_BIN_OUTPUT_DIR = Path('/content/drive/MyDrive/wood-moisture-2d-cnn-outputs-moisture-bins')
MOISTURE_BIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['MOISTURE_BIN_OUTPUT_DIR'] = str(MOISTURE_BIN_OUTPUT_DIR)
model_output_dir = os.environ.get('OUTPUT_DIR', '/content/drive/MyDrive/wood-moisture-2d-cnn-outputs')
print('model source OUTPUT_DIR =', model_output_dir)
print('moisture-bin output dir =', MOISTURE_BIN_OUTPUT_DIR)
print('moisture-bin output dir exists =', MOISTURE_BIN_OUTPUT_DIR.exists())
!python -m src.interpret.run_moisture_bin_interpretability --tasks mc species woodtype wood_structure index_norm mc_norm --model-output-dir "{model_output_dir}" --output-dir "{MOISTURE_BIN_OUTPUT_DIR}" --binning fixed --max-samples-per-bin 96 --ig-steps 24
print('saved files under moisture-bin output dir:')
for path in sorted(MOISTURE_BIN_OUTPUT_DIR.rglob('*'))[:80]:
    print(path.relative_to(MOISTURE_BIN_OUTPUT_DIR))
